# Pinecone Indexing 생성

In [13]:
from dotenv import load_dotenv
load_dotenv()

PINECONE_INDEX_NAME = 'adv-rag'
PINECONE_INDEX_REGION = 'us-east-1'
PINECONE_INDEX_CLOUD = 'aws'
PINECONE_INDEX_METRIc = 'cosine'
PINECONE_INDEX_DEMENSION = 1536

OPENAI_LLM_MODEL = 'gpt-4.1-mini'
OPENAI_EMBEDDING_MODEL = 'text-embedding-3-small'

In [2]:
from pinecone import Pinecone,ServerlessSpec

pc = Pinecone()
print(pc.list_indexes().names())

if PINECONE_INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=PINECONE_INDEX_DEMENSION,
        metric=PINECONE_INDEX_METRIc,
        spec=ServerlessSpec(
            region=PINECONE_INDEX_REGION,
            cloud=PINECONE_INDEX_CLOUD
        )
    )
    print(f'{PINECONE_INDEX_NAME} index 생성 완료')
else:
    print('{PINECONE_INDEX_NAME} index가 이미 존재합니다.')

['adv-rag', 'winemeg-review-data']
{PINECONE_INDEX_NAME} index가 이미 존재합니다.


In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_pinecone import PineconeVectorStore

# 임베딩 모델 생성
embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL)

# 벡터 스토어 생성
vector_store = PineconeVectorStore(
    index_name=PINECONE_INDEX_NAME,
    embedding=embeddings
)

In [15]:
import pandas as pd

document_df = pd.read_csv('data/documents.csv')
document_df

,doc_id,title,content
0,D1,제주도 여행 가이드,"제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(..."
1,D2,전주 비빔밥과 진주 비빔밥 차이점,"비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기..."
2,D3,걸스데이 히트곡 분석,"걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Somet..."
3,D4,세종대왕과 훈민정음,세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입...
4,D5,이순신 장군의 명량 해전,이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133...
5,D6,2024년 기후 변화 종합 보고서,"2024년 전 지구 평균 기온은 산업화 이전 대비 약 1.2℃ 상승했으며, 해수면 ..."
6,D7,AI 기술 동향 및 윤리,"최근 인공지능 분야에서는 생성형 AI, 멀티모달 모델, 강화학습 기반 에이전트 개발..."
7,D8,서울 지하철 이용 가이드,"서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로..."
8,D9,판소리 “춘향가” 서사 구조,"판소리는 소리꾼과 고수가 함께 공연하는 한국 전통 음악으로, 대표 작품에는 “춘향가..."
9,D10,한국 축구 대표팀 주요 기록,"한국 축구 대표팀은 2002 한일 월드컵 4강 진출, 2012 런던 올림픽 동메달 ..."


In [16]:
docs_to_index = []
for idx,row in document_df.iterrows():
    doc_id = row['doc_id']
    content = row['content']
    docs_to_index.append((doc_id,content))

docs_to_index

[('D1',
  '제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.'),
 ('D2',
  '비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기·계란 등)을 올리고 고추장이나 간장을 섞어 먹습니다. 전주 비빔밥은 고명 종류가 다양하고 전주식 고추장을 쓰며, 잔치용으로도 유명합니다. 진주 비빔밥은 고기·회·나물 등을 섞어 더욱 풍부한 식감을 제공합니다. 두 지역 모두 역사적 배경과 재료 구성이 달라 맛과 풍미가 다릅니다.'),
 ('D3',
  '걸스데이는 2010년 데뷔한 대한민국의 4인조 걸그룹으로, 대표곡으로는 “Something”, “Darling”, “Expectation” 등이 있습니다. 데뷔 초기 청순 컨셉에서 점차 섹시·여성미 컨셉으로 변화하며 음원 차트 상위권에 올랐습니다. 멤버 민아·유라·소진·혜리는 드라마·예능·광고 등 다양한 분야에도 진출해 활동 영역을 넓혔습니다.'),
 ('D4',
  '세종대왕(1397~1450)은 훈민정음을 창제하여 한글을 보급한 조선의 4대 임금입니다. 그가 훈민정음을 만든 배경에는 백성들의 문맹 문제 해결과 국가 통치 효율화가 있었습니다. 세종대왕의 업적은 한국 문화와 문자 체계에 지대한 영향을 미쳤으며, 훈민정음 해례본은 유네스코 세계기록유산으로 등재되었습니다.'),
 ('D5',
  '이순신 장군(1545~1598)은 임진왜란 당시 명량 해전에서 13척의 배로 133척의 왜선을 격파하면서 크게 승리했습니다. 전술적인 배 배치(학익진)와 기상·해류를 활용한 전략은 전투 역사에 길이 남을 전술입니다. 이순신의 업적은 한국 해군 전통과 군사 전략 연구에서 핵심 사례로 다뤄집니다.'),
 ('D6

In [17]:
from langchain_core.documents import Document

docs = []
ids = []

for idx,row in document_df.iterrows():
    doc_id = row['doc_id']
    content = row['content']
    
    doc = Document(
        page_content=content,
        metadata = {
            "doc_id":doc_id
        }
    )

    docs.append(doc)
    ids.append(doc_id)

vector_store.add_documents(
    documents=docs,
    ids=ids
)

print("Pinecone 문서 저장 완료")
print(vector_store._index.describe_index_stats())

Pinecone 문서 저장 완료
{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 30}},
 'total_vector_count': 30,
 'vector_type': 'dense'}


## 유사도 검색 수행

In [19]:
query = "제주도 관광지"

results = vector_store.similarity_search_with_score(query,k=5)

for rank, (doc, score) in enumerate(results,start=1):
    print(f"{rank} : {doc.metadata["doc_id"]} ({score})")
    print(f"{doc.page_content}")

1 : D1 (0.589187562)
제주도는 대한민국의 대표 관광지로서, 한라산 등반, 성산 일출봉 관광, 해변 활동(협재해수욕장·함덕해수욕장) 등이 인기입니다. 현지 음식으로는 흑돼지, 고기국수, 전복죽 등이 있으며, 카페 거리(서귀포시 대정읍 카페 거리)도 유명합니다. 교통은 렌터카나 시외버스를 주로 이용하며, 사전 예약 시 우도 투어나 올레길 트레킹도 즐길 수 있습니다.
2 : D12 (0.309271365)
서울 근교에서 당일치기로 다녀올 만한 여행지로는 가평 쁘띠프랑스, 남양주 수종사, 양평 두물머리, 용인 에버랜드 등이 있습니다. 기차·버스 노선이 잘 발달되어 있어 대중교통으로 이동이 편리하며, 차가 있다면 경춘고속도로를 이용해 접근성이 좋습니다. 사전 관광 예약 앱(예: 야놀자, 쿠팡트래블)에서도 할인 혜택을 확인할 수 있습니다.
3 : D13 (0.230494425)
비빔밥은 지역별로 칼로리, 탄수화물, 단백질, 지방 함량이 차이를 보입니다. 전주 비빔밥(약 650kcal)은 채소·고기·계란 비율이 고르지만, 진주 비빔밥(약 700kcal)은 해산물과 육류가 섞여 열량이 다소 높습니다. 안동 비빔밥은 재료가 비교적 간단해 600kcal 내외이며, 지역별 나물 종류와 기름 사용량이 칼로리 차이에 영향을 미칩니다.
4 : D2 (0.227553785)
비빔밥은 조선 시대부터 전해 내려온 대표적 한국 음식으로, 밥 위에 고명(채소·고기·계란 등)을 올리고 고추장이나 간장을 섞어 먹습니다. 전주 비빔밥은 고명 종류가 다양하고 전주식 고추장을 쓰며, 잔치용으로도 유명합니다. 진주 비빔밥은 고기·회·나물 등을 섞어 더욱 풍부한 식감을 제공합니다. 두 지역 모두 역사적 배경과 재료 구성이 달라 맛과 풍미가 다릅니다.
5 : D8 (0.19694528)
서울 지하철은 1호선부터 9호선까지 운행되며, 주요 환승역으로는 서울역·강남역·종로3가역·고속터미널역 등이 있습니다. 기본 요금은 1,250원(성인 기준)이며, 거리에 따른 추가 요금이 부과됩니다. T-mon